# Final baseline run: fine-tuned encoders with an honest, resume-safe training time

**Goal.** Re-train the fine-tuned baselines from scratch and record a *correct* training time, even if Kaggle stops the session and we resume.

**Dataset.** `imbalance_free_dataset` only (train 4,050 = 1,350 per class, validation 333, test 333). This is the same training data that VBRI stores, so the comparison is controlled. The notebook refuses to run on any other dataset.

**How the training time is kept correct**
* A timer callback measures active training time (`perf_counter`) and writes a heartbeat to disk every 20 s.
* Every checkpoint gets a `timing_at_checkpoint.json` with the total active time up to that point.
* After a crash the run resumes from the last *valid* checkpoint and continues from the time stored in that checkpoint, not from zero and not from Hugging Face's `train_runtime` (which only counts the resumed part).
* Per run we report: **effective time** (time of an uninterrupted run, use this in the paper), **gross time** (all compute spent, including work lost in a crash) and **wasted time** (gross minus effective).
* A run is skipped only if it was completed by *this* code (`result.json` with the same `CODE_VERSION`). Old saved models are never reused.

**Kaggle settings:** Accelerator = GPU T4 (one GPU is used), Persistence = *Files only* (or more), Internet = On.
**If the session dies:** open the notebook again and press *Run All*. Finished runs are skipped, the interrupted run resumes from its last checkpoint.

**Stages:** 1 setup -> 2 smoke test (crash and resume check) -> 3 training (seeds x models) -> 4 summary tables -> 5 inference benchmark -> 6 VBRI benchmark (same session, same hardware) -> 7 zip results.

## 1. Setup and settings

In [ ]:
import os, sys, json, time, shutil, hashlib, platform, subprocess, random, math, gc, re, csv, statistics, zipfile, inspect, traceback
from pathlib import Path
from datetime import datetime, timezone

# Use ONE GPU so that timings are not affected by Kaggle's 2xT4 / DataParallel. Must be set before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import contractions
except ImportError:
    _pip("contractions")
    import contractions
try:
    import sentence_transformers  # needed only for the VBRI benchmark
except ImportError:
    _pip("sentence-transformers")

# ------------------------------ settings ------------------------------------
CODE_VERSION = "final_baseline_v1"
SEEDS = [42, 43, 44]                       # run order: all models for seed 42 first, then 43, then 44
MODELS = ["ELECTRA", "BERT", "DistilBERT", "RoBERTa", "XLNet", "DeBERTa", "ModernBERT"]
MAX_EPOCHS = 15
PATIENCE = 2                               # early stopping on validation loss
RUN_SMOKE_TEST = True                      # short crash-and-resume check (skipped automatically once it has passed)
RUN_MAIN_TRAINING = True
RUN_INFERENCE_BENCH = True
RUN_VBRI_BENCH = True
KEEP_WEIGHTS_SEEDS = [SEEDS[0]]            # weights are kept only for these seeds (needed for the inference benchmark)
FORCE_FRESH = False                        # DANGER: True deletes everything this notebook produced. Leave False when resuming.
DATASET_DIR_OVERRIDE = None                # e.g. "/kaggle/input/<name>/imbalance_free_dataset"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working/final_baseline_run") if ON_KAGGLE else Path("./final_baseline_run_out")
if FORCE_FRESH and WORK_DIR.exists():
    print("!!! FORCE_FRESH=True -> deleting", WORK_DIR)
    shutil.rmtree(WORK_DIR)
RUNS_DIR = WORK_DIR / "runs"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("Work dir:", WORK_DIR.resolve())
print("Free disk (GB):", round(shutil.disk_usage(WORK_DIR).free / 1e9, 1))

In [ ]:
import torch, transformers, datasets
import numpy as np

cuda_ok = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if cuda_ok else "cpu"
CAP = torch.cuda.get_device_capability(0) if cuda_ok else (0, 0)
# T4 (compute capability 7.5) has no real bf16 -> fp16 for every model. bf16 only on Ampere+ (cc >= 8).
PRECISION = "bf16" if CAP[0] >= 8 else ("fp16" if cuda_ok else "fp32")

ENV = {
    "code_version": CODE_VERSION, "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "gpu": GPU_NAME, "compute_capability": list(CAP), "precision": PRECISION,
    "visible_gpus": torch.cuda.device_count() if cuda_ok else 0,
    "torch": torch.__version__, "transformers": transformers.__version__, "datasets": datasets.__version__,
    "python": platform.python_version(), "cpu_count": os.cpu_count(), "platform": platform.platform(),
}
print(json.dumps(ENV, indent=2))
if not cuda_ok:
    print("\n!!! No GPU found. Training times will not be meaningful. Enable a GPU accelerator in Kaggle.")
elif "T4" not in GPU_NAME:
    print(f"\n!!! GPU is '{GPU_NAME}', not a Tesla T4. Times are only comparable with other runs on the same GPU type.")
(WORK_DIR / "env.json").write_text(json.dumps(ENV, indent=2))

## 2. Helpers: JSON, metrics, fingerprints (pure Python)

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def atomic_write_json(path, obj):
    """Write JSON so that a crash can never leave a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default

CLASSES = ["Normal", "BPD", "Inconclusive"]

def confusion(y, p, k=3):
    cm = [[0] * k for _ in range(k)]
    for a, b in zip(y, p):
        cm[int(a)][int(b)] += 1
    return cm

def summarize(y, p):
    """All metrics in %, computed from labels y and predictions p (0=Normal, 1=BPD, 2=Inconclusive)."""
    cm = confusion(y, p)
    n = len(y)
    per = []
    for c in range(3):
        tp = cm[c][c]
        fp = sum(cm[i][c] for i in range(3)) - tp
        fn = sum(cm[c]) - tp
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per.append((pr, rc, f1, sum(cm[c])))
    tp = cm[1][1]; fn = sum(cm[1]) - tp
    fp = sum(cm[i][1] for i in (0, 2)); tn = n - tp - fn - fp
    safe = lambda a, b: 100.0 * a / b if b else 0.0
    return {
        "n": n, "confusion": cm,
        "accuracy": 100.0 * sum(cm[i][i] for i in range(3)) / n,
        "macro_f1": 100.0 * sum(x[2] for x in per) / 3,
        "weighted_f1": 100.0 * sum(x[2] * x[3] for x in per) / n,
        "bpd_f1": 100.0 * per[1][2], "bpd_precision": 100.0 * per[1][0],
        "bpd_sensitivity": 100.0 * per[1][1], "inconclusive_recall": 100.0 * per[2][1],
        "bin_sensitivity": safe(tp, tp + fn), "bin_specificity": safe(tn, tn + fp),
        "bin_ppv": safe(tp, tp + fp), "bin_npv": safe(tn, tn + fn),
        "bin_accuracy": safe(tp + tn, n),
        "bin_dor": (tp * tn) / (fp * fn) if fp * fn else float("nan"),
    }

def auroc(scores, ybin):
    """AUROC by the rank (Mann-Whitney) method, ties get average rank."""
    n = len(scores)
    n_pos = sum(1 for y in ybin if y); n_neg = n - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = sorted(range(n), key=lambda i: scores[i])
    ranks = [0.0] * n
    i = 0
    while i < n:
        j = i
        while j + 1 < n and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j) / 2 + 1
        for t in range(i, j + 1):
            ranks[order[t]] = avg
        i = j + 1
    r_pos = sum(ranks[i] for i in range(n) if ybin[i])
    return (r_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

def spec_at_sens(scores, ybin, target=0.85):
    """Specificity at the highest threshold whose sensitivity reaches `target`."""
    P = sum(1 for y in ybin if y); N = len(ybin) - P
    if P == 0 or N == 0:
        return float("nan"), None
    for t in sorted(set(scores), reverse=True):
        tp = sum(1 for s, y in zip(scores, ybin) if s >= t and y)
        fp = sum(1 for s, y in zip(scores, ybin) if s >= t and not y)
        if tp / P >= target:
            return 100.0 * (1 - fp / N), t
    return 0.0, None

def fingerprint(rows, keys=("question_id", "label", "answer")):
    lines = sorted("|".join(str(r[k]).strip() for k in keys) for r in rows)
    return hashlib.sha256("\n".join(lines).encode("utf-8")).hexdigest()

def pct_stats(vals):
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and math.isnan(v))]
    if not vals:
        return float("nan"), float("nan")
    return statistics.mean(vals), (statistics.stdev(vals) if len(vals) > 1 else 0.0)

## 3. Resume-safe timer

`RunTimer` is a Trainer callback. It keeps the numbers that matter in files on disk, so they survive a Kaggle restart:

| file | content |
|---|---|
| `ledger.json` | one entry per session: start time, GPU, active seconds (heartbeat every 20 s), status (`running`, `interrupted`, `crashed`, `completed`) and a summary |
| `checkpoint-N/timing_at_checkpoint.json` | cumulative effective training time at the moment checkpoint N was saved |
| `train_complete.json` | written when training finished normally |

Definitions: **effective** = time stored in the checkpoint we resumed from + active time after the resume. **gross** = sum of the active time of all sessions. **wasted** = gross - effective (the partial epoch lost in a crash). Evaluation and checkpoint-saving time are part of training time (as in Hugging Face's own `train_runtime`) and are also reported separately.

In [ ]:
try:
    from transformers import TrainerCallback
except Exception:                      # lets the timing logic be unit-tested without transformers
    class TrainerCallback:
        pass


class RunTimer(TrainerCallback):
    """Active-training-time bookkeeping that survives crashes and resumes."""

    def __init__(self, run_dir, base=None, resumed_from_step=0, meta=None, heartbeat_s=20.0):
        self.run_dir = Path(run_dir)
        self.ledger_path = self.run_dir / "ledger.json"
        base = base or {}
        self.base_effective = float(base.get("effective_s", 0.0))
        self.base_eval = float(base.get("eval_s", 0.0))
        self.base_save = float(base.get("save_s", 0.0))
        self.resumed_from_step = int(resumed_from_step)
        self.meta = dict(meta or {})
        self.heartbeat_s = float(heartbeat_s)
        self.eval_s = 0.0            # evaluation time in this session
        self.save_s = 0.0            # checkpoint-saving time in this session
        self._t0 = None
        self._frozen = None
        self._last_hb = 0.0
        self._open = False
        self._ledger = None

    # ---- accessors
    def active_s(self):
        if self._t0 is None:
            return 0.0
        return self._frozen if self._frozen is not None else time.perf_counter() - self._t0

    def effective_s(self):
        return self.base_effective + self.active_s()

    def eval_total_s(self):
        return self.base_eval + self.eval_s

    def save_total_s(self):
        return self.base_save + self.save_s

    # ---- ledger
    def _load_ledger(self):
        led = read_json(self.ledger_path, None)
        if not isinstance(led, dict) or "sessions" not in led:
            led = {"code_version": CODE_VERSION, "created_utc": utc_now(), "sessions": []}
        return led

    def _heartbeat(self, status=None, step=None, epoch=None):
        s = self._ledger["sessions"][-1]
        s["active_s"] = round(self.active_s(), 3)
        s["effective_s_so_far"] = round(self.effective_s(), 3)
        s["last_heartbeat_utc"] = utc_now()
        if step is not None:
            s["last_step"] = int(step)
        if epoch is not None:
            s["last_epoch"] = float(epoch)
        if status:
            s["status"] = status
        atomic_write_json(self.ledger_path, self._ledger)
        self._last_hb = time.perf_counter()

    @staticmethod
    def summarize_ledger(led, effective_s, eval_s, save_s):
        gross = sum(float(s.get("active_s", 0.0)) for s in led["sessions"])
        return {
            "effective_s": round(effective_s, 3), "eval_s": round(eval_s, 3), "save_s": round(save_s, 3),
            "gross_s": round(gross, 3), "wasted_s": round(max(0.0, gross - effective_s), 3),
            "n_sessions": len(led["sessions"]),
            "resumed": any(s.get("resumed_from_step", 0) > 0 for s in led["sessions"]),
            "session_status": [s.get("status") for s in led["sessions"]],
        }

    def close(self, status, step=None, epoch=None):
        """Finish this session (status: completed / crashed). Safe to call twice."""
        if not self._open:
            return
        self._frozen = time.perf_counter() - self._t0
        self._heartbeat(status=status, step=step, epoch=epoch)
        self._ledger["summary"] = self.summarize_ledger(
            self._ledger, self.effective_s(), self.eval_total_s(), self.save_total_s())
        atomic_write_json(self.ledger_path, self._ledger)
        self._open = False
        if status == "completed":
            atomic_write_json(self.run_dir / "train_complete.json",
                              {"finished_utc": utc_now(), **self._ledger["summary"]})

    # ---- Trainer callback hooks
    def on_train_begin(self, args, state, control, **kwargs):
        led = self._load_ledger()
        for s in led["sessions"]:                      # a session still marked running was killed
            if s.get("status") == "running":
                s["status"] = "interrupted"
        led.pop("summary", None)
        self._t0 = time.perf_counter()
        self._frozen = None
        led["sessions"].append({
            "session": len(led["sessions"]) + 1, "started_utc": utc_now(),
            "resumed_from_step": self.resumed_from_step,
            "base_effective_s": round(self.base_effective, 3),
            "active_s": 0.0, "status": "running", **self.meta,
        })
        self._ledger = led
        self._open = True
        self._heartbeat()
        return control

    def on_step_end(self, args, state, control, **kwargs):
        if self._open and time.perf_counter() - self._last_hb >= self.heartbeat_s:
            self._heartbeat(step=state.global_step, epoch=state.epoch)
        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        if self._open:
            self._heartbeat(step=state.global_step, epoch=state.epoch)
        return control

    def on_save(self, args, state, control, **kwargs):
        if not self._open:
            return control
        ck = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ck.exists():
            atomic_write_json(ck / "timing_at_checkpoint.json", {
                "global_step": int(state.global_step), "epoch": float(state.epoch or 0.0),
                "effective_s": round(self.effective_s(), 3),
                "eval_s": round(self.eval_total_s(), 3), "save_s": round(self.save_total_s(), 3),
                "written_utc": utc_now(), "session": len(self._ledger["sessions"]),
            })
        self._heartbeat(step=state.global_step, epoch=state.epoch)
        return control

    def on_train_end(self, args, state, control, **kwargs):
        self.close("completed", step=state.global_step, epoch=state.epoch)
        return control


# ---------------------------- checkpoint helpers ------------------------------
def list_checkpoints(run_dir):
    cks = [p for p in Path(run_dir).glob("checkpoint-*") if p.is_dir() and p.name.split("-")[-1].isdigit()]
    return sorted(cks, key=lambda p: int(p.name.split("-")[-1]), reverse=True)

def checkpoint_ok(ck):
    """A checkpoint is usable only if weights, trainer state and OUR timing file are all present and readable."""
    ck = Path(ck)
    has_weights = any((ck / n).exists() for n in
                      ("model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"))
    st = read_json(ck / "trainer_state.json", None)
    tm = read_json(ck / "timing_at_checkpoint.json", None)
    return bool(has_weights and isinstance(st, dict) and isinstance(tm, dict) and "effective_s" in tm)

def pick_resume_checkpoint(run_dir, log=print):
    """Latest valid checkpoint (and its timing). Invalid newer ones (e.g. half-written when Kaggle died) are deleted."""
    for ck in list_checkpoints(run_dir):
        if checkpoint_ok(ck):
            return ck, read_json(ck / "timing_at_checkpoint.json")
        log(f"  [resume] checkpoint {ck.name} is incomplete or has no timing file -> deleting it")
        shutil.rmtree(ck, ignore_errors=True)
    return None, None

## 4. Dataset: `imbalance_free_dataset` only

In [ ]:
EXPECTED = {
    "train_rows": 4050, "val_rows": 333, "test_rows": 333,
    "train_per_class": 1350, "test_label_counts": {0: 225, 1: 67, 2: 41},
    "train_fp": "529c5eab3547f1e7f7c4240444adcd6296c377d141ac6e275b488e066e1b477f",
    "test_fp": "b0d1f78b4093c5ef95ad488a014ddf09c5c161e4953f786e79341c7085578941",
}

def find_imbalance_free_dir():
    if DATASET_DIR_OVERRIDE:
        return Path(DATASET_DIR_OVERRIDE), [Path(DATASET_DIR_OVERRIDE)]
    found = []
    for root in [Path("/kaggle/input"), Path("."), Path("experiments/dataset"), Path("../experiments/dataset"), Path("./dataset")]:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            d = Path(dirpath)
            if len(d.relative_to(root).parts) > 5:
                dirnames[:] = []
                continue
            if d.name == "imbalance_free_dataset" and (d / "dataset_dict.json").exists():
                found.append(d)
                dirnames[:] = []
    if not found:
        raise FileNotFoundError("imbalance_free_dataset not found. Attach the Kaggle dataset or set DATASET_DIR_OVERRIDE.")
    return found[0], found

from datasets import load_from_disk, Dataset, DatasetDict

DATA_DIR, _all = find_imbalance_free_dir()
print("Candidates found:", [str(p) for p in _all])
print("USING DATASET   :", DATA_DIR)
RAW = load_from_disk(str(DATA_DIR))

# ---- hard checks: stop if this is not the dataset the paper describes -------------
assert len(RAW["train"]) == EXPECTED["train_rows"], f"train rows {len(RAW['train'])} != {EXPECTED['train_rows']}"
assert len(RAW["validation"]) == EXPECTED["val_rows"] and len(RAW["test"]) == EXPECTED["test_rows"], "val/test size mismatch"
tr_rows = RAW["train"].to_pandas().to_dict("records")
te_rows = RAW["test"].to_pandas().to_dict("records")
from collections import Counter
tr_counts = Counter(int(r["label"]) for r in tr_rows)
assert all(tr_counts[c] == EXPECTED["train_per_class"] for c in (0, 1, 2)), f"train not balanced: {tr_counts}"
q_counts = Counter((int(r["question_id"]), int(r["label"])) for r in tr_rows)
assert all(v == 150 for v in q_counts.values()) and len(q_counts) == 27, "train is not 150 per class per question"
assert dict(Counter(int(r["label"]) for r in te_rows)) == EXPECTED["test_label_counts"], "test label counts differ"
fp_tr, fp_te = fingerprint(tr_rows), fingerprint(te_rows)
assert fp_tr == EXPECTED["train_fp"], "TRAIN fingerprint differs from the verified imbalance_free_dataset"
assert fp_te == EXPECTED["test_fp"], "TEST fingerprint differs: the test set is not the one used for VBRI"
print("OK: train 4050 (1350/class, 150/class/question), validation 333, test 333 (225/67/41). Fingerprints match.")

DATASET_REPORT = {"path": str(DATA_DIR), "train_fp": fp_tr, "test_fp": fp_te,
                  "train": len(RAW["train"]), "validation": len(RAW["validation"]), "test": len(RAW["test"])}
atomic_write_json(WORK_DIR / "dataset_used.json", DATASET_REPORT)

# keep test info aligned with prediction order
TEST_IDS = [int(r["id"]) for r in te_rows]
TEST_QIDS = [int(r["question_id"]) for r in te_rows]
TEST_Y = [int(r["label"]) for r in te_rows]

In [ ]:
class TextNormalizer:
    def __init__(self):
        # Initialize text normalization dictionary
        self.slang_dict = {
            'imo': 'in my opinion',
            'imho': 'in my humble opinion',
            'tbh': 'to be honest',
            'afaik': 'as far as i know',
            'idk': "i don't know",
            'smh': 'shaking my head',
            'btw': 'by the way',
            'asap': 'as soon as possible',
            'fyi': 'for your information',
            'np': 'no problem',
            'thx': 'thanks',
            'pls': 'please',
            'u': 'you',
            'r': 'are',
            'ur': 'your',
            'n': 'and',
            'b/c': 'because',
            'cz' : 'because',
            'bcz' : 'because',
            'ofc' : 'ofcourse',
            'govt': 'government',
            'dept': 'department',
            'est': 'established',
            'approx': 'approximately',
            'min': 'minimum',
            'max': 'maximum',
            "n't": 'not',
            "nt" : 'not',
            "gn" : 'good night',
            "f9" : 'fine',
            'bpd': 'borderline personality disorder',
            'dbt': 'dialectical behavior therapy',
            'fp': 'favorite person',
            'si': 'suicidal ideation',
            'sh': 'self harm',
            "its" : 'it is',
            "people's" : 'people',
            "sha'n't": "shall not",
            "shan't've": "shall not have",
            "she'd": "she had / she would",
            "she'd've": "she would have",
            "she'll": "she shall / she will",
            "she'll've": "she shall have / she will have",
            "she's": "she has / she is",
            "should've": "should have",
            "shouldn't": "should not",
            "shouldn't've": "should not have",
            "so've": "so have",
            "so's": "so as / so is",
            "that'd": "that would / that had",
            "that'd've": "that would have",
            "that's": "that has / that is",
            "there'd": "there had / there would",
            "there'd've": "there would have",
            "there's": "there has / there is",
            "they'd": "they had / they would",
            "they'd've": "they would have",
            "they'll": "they shall / they will",
            "they'll've": "they shall have / they will have",
            "they're": "they are",
            "they've": "they have",
            "to've": "to have",
            "wasn't": "was not",
            "we'd": "we had / we would",
            "we'd've": "we would have",
            "we'll": "we will",
            "we'll've": "we will have",
            "we're": "we are",
            "we've": "we have",
            "weren't": "were not",
            "what'll": "what shall / what will",
            "what'll've": "what shall have / what will have",
            "what're": "what are",
            "what's": "what has / what is",
            "what've": "what have",
            "when's": "when has / when is",
            "when've": "when have",
            "where'd": "where did",
            "where's": "where has / where is",
            "where've": "where have",
            "who'll": "who shall / who will",
            "who'll've": "who shall have / who will have",
            "who's": "who has / who is",
            "who've": "who have",
            "why's": "why has / why is",
            "why've": "why have",
            "will've": "will have",
            "won't": "will not",
            "won't've": "will not have",
            "would've": "would have",
            "wouldn't": "would not",
            "wouldn't've": "would not have",
            "y'all": "you all",
            "y'all'd": "you all would",
            "y'all'd've": "you all would have",
            "y'all're": "you all are",
            "y'all've": "you all have",
            "you'd": "you had / you would",
            "you'd've": "you would have",
            "you'll": "you shall / you will",
            "you'll've": "you shall have / you will have",
            "you're": "you are",
            "you've": "you have"
}

    def clean_text(self, text):
        """Basic text cleaning"""
        text = str(text).lower().strip()
        # Remove special characters (keep basic punctuation)
        text = re.sub(r"[^a-zA-Z0-9\s.,!?']", "", text)
        return text

    def expand_slang(self, text):
        """Replace slang terms using dictionary"""
        words = text.split()
        expanded_words = [self.slang_dict.get(word, word) for word in words]
        return ' '.join(expanded_words)

    def normalize_text(self, text):
        """Full text normalization pipeline"""
        if not isinstance(text, str) or not text.strip():
            return text
            
        # Clean text first
        text = self.clean_text(text)
        # Fix contractions
        text = contractions.fix(text)
        # Expand slang
        text = self.expand_slang(text)
        return text


# ---- same text cleaning as the earlier baseline (lower-case, special characters, contractions, slang) ----
normalizer = TextNormalizer()
def _normalise_split(ds):
    df = ds.to_pandas()
    for col in ("question", "answer"):
        df[col] = df[col].apply(normalizer.normalize_text)
    return Dataset.from_pandas(df, preserve_index=False)

CLEAN = DatasetDict({s: _normalise_split(RAW[s]) for s in ("train", "validation", "test")})
TEST_Q_NORM, TEST_A_NORM = list(CLEAN["test"]["question"]), list(CLEAN["test"]["answer"])
print("Example (normalised):", CLEAN["train"][0]["answer"][:120])

## 5. Model settings, trainer and the training function

Hyper-parameters are the same as in the earlier baseline notebook (effective batch size 16 for every model).
Only validation loss selects the checkpoint; the test set is scored once, after training.

In [ ]:
MODEL_SPECS = {
    "ELECTRA":    dict(checkpoint="google/electra-base-discriminator", bs=16, eval_bs=16, accum=1, lr=5e-6, wd=0.1,  warmup=0.2),
    "BERT":       dict(checkpoint="bert-base-uncased",                 bs=16, eval_bs=16, accum=1, lr=5e-6, wd=0.1,  warmup=0.2),
    "DistilBERT": dict(checkpoint="distilbert/distilbert-base-uncased", bs=16, eval_bs=16, accum=1, lr=5e-6, wd=0.1,  warmup=0.2),
    "RoBERTa":    dict(checkpoint="FacebookAI/roberta-base",           bs=16, eval_bs=16, accum=1, lr=5e-6, wd=0.1,  warmup=0.2),
    "XLNet":      dict(checkpoint="xlnet/xlnet-base-cased",            bs=4,  eval_bs=2,  accum=4, lr=5e-6, wd=0.1,  warmup=0.2, eval_accum=8, pad_left=True),
    "DeBERTa":    dict(checkpoint="microsoft/deberta-v3-base",         bs=8,  eval_bs=8,  accum=2, lr=2e-5, wd=0.01, warmup=0.1),
    "ModernBERT": dict(checkpoint="answerdotai/ModernBERT-base",       bs=8,  eval_bs=8,  accum=2, lr=2e-5, wd=0.01, warmup=0.1),
}

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, EarlyStoppingCallback, set_seed)

def _logits(pred):
    p = pred.predictions if hasattr(pred, "predictions") else pred
    return p[0] if isinstance(p, tuple) else p

def compute_metrics_hf(eval_pred):
    logits = _logits(eval_pred)
    y = eval_pred.label_ids
    m = summarize([int(v) for v in y], [int(v) for v in np.argmax(logits, axis=-1)])
    return {"accuracy": m["accuracy"] / 100.0, "f1": m["weighted_f1"] / 100.0}


class TimedTrainer(Trainer):
    """Trainer that also reports how much time went into evaluation and checkpoint saving."""
    timer = None

    def evaluate(self, *a, **k):
        t = time.perf_counter()
        out = super().evaluate(*a, **k)
        if self.timer is not None and self.timer._open:
            self.timer.eval_s += time.perf_counter() - t
        return out

    def _save_checkpoint(self, *a, **k):
        t = time.perf_counter()
        out = super()._save_checkpoint(*a, **k)
        if self.timer is not None and self.timer._open:
            self.timer.save_s += time.perf_counter() - t
        return out


def build_training_args(spec, seed, run_dir, epochs, precision):
    params = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(
        output_dir=str(run_dir),
        per_device_train_batch_size=spec["bs"], per_device_eval_batch_size=spec["eval_bs"],
        gradient_accumulation_steps=spec["accum"], num_train_epochs=epochs,
        learning_rate=spec["lr"], weight_decay=spec["wd"], lr_scheduler_type="linear",
        save_strategy="epoch", logging_strategy="epoch",
        save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False,
        seed=seed, data_seed=seed, report_to="none",
        fp16=(precision == "fp16"), bf16=(precision == "bf16"),
    )
    kw["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "epoch"
    if "warmup_ratio" in params:
        kw["warmup_ratio"] = spec["warmup"]
    else:
        kw["warmup_steps"] = spec["warmup"]          # newer transformers accept a float ratio here
    if spec.get("eval_accum"):
        kw["eval_accumulation_steps"] = spec["eval_accum"]
    return TrainingArguments(**kw)


_TOKEN_CACHE = {}
def get_tokenised(model_name, subset=None):
    spec = MODEL_SPECS[model_name]
    if model_name not in _TOKEN_CACHE:
        tok = AutoTokenizer.from_pretrained(spec["checkpoint"])
        if spec.get("pad_left"):
            tok.padding_side = "left"                 # XLNet needs left padding
        def f(ex):
            return tok(ex["question"], ex["answer"], truncation=True)   # question + answer as a pair, dynamic padding later
        out = DatasetDict()
        for split in ("train", "validation", "test"):
            ds = CLEAN[split]
            out[split] = ds.map(f, batched=True, remove_columns=[c for c in ds.column_names if c != "label"])
        _TOKEN_CACHE[model_name] = (tok, out)
    tok, out = _TOKEN_CACHE[model_name]
    if subset:                                         # smoke test only
        tr = out["train"].shuffle(seed=0).select(range(subset["train"]))
        va = out["validation"].select(range(subset["val"]))
        out = DatasetDict({"train": tr, "validation": va, "test": out["test"]})
    return tok, out


def softmax_rows(x):
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return (e / e.sum(axis=1, keepdims=True)).tolist()


def write_test_probs(path, probs):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "question_id", "label", "p_normal", "p_bpd", "p_inconclusive"])
        for i, (pid, q, y, p) in enumerate(zip(TEST_IDS, TEST_QIDS, TEST_Y, probs)):
            w.writerow([pid, q, y, *[f"{v:.6f}" for v in p]])


def _last_state(run_dir):
    for ck in list_checkpoints(run_dir):
        st = read_json(ck / "trainer_state.json", None)
        if st:
            return st
    return None


def run_training(model_name, seed, variant="", precision=None, run_root=None, epochs=None, patience=None,
                 subset=None, crash_at_step=None, simulate_kill=False, heartbeat_s=20.0, keep_weights=None):
    """Train (or resume, or load a finished run of this code version). Returns the run's result dict."""
    spec = MODEL_SPECS[model_name]
    precision = precision or PRECISION
    epochs = epochs or MAX_EPOCHS
    patience = PATIENCE if patience is None else patience
    run_root = Path(run_root) if run_root else RUNS_DIR
    run_dir = run_root / f"seed{seed}" / f"{model_name}{variant}"
    run_dir.mkdir(parents=True, exist_ok=True)
    keep_weights = (seed in KEEP_WEIGHTS_SEEDS) if keep_weights is None else keep_weights
    tag = f"seed{seed}/{model_name}{variant}"

    done = read_json(run_dir / "result.json", None)
    if done and done.get("code_version") == CODE_VERSION and done.get("status") == "done":
        print(f"[{tag}] already finished by this notebook -> skipped (effective time {done['timing']['effective_s']:.0f}s)")
        return done

    print(f"\n{'='*78}\n[{tag}] precision={precision}  gpu={GPU_NAME}\n{'='*78}")
    set_seed(seed)
    tok, ds = get_tokenised(model_name, subset)
    model = AutoModelForSequenceClassification.from_pretrained(spec["checkpoint"], num_labels=3)
    if spec.get("pad_left"):
        model.config.pad_token_id = tok.pad_token_id
    n_params = sum(p.numel() for p in model.parameters())

    ck, tm = pick_resume_checkpoint(run_dir)
    train_done = (run_dir / "train_complete.json").exists() and ck is not None
    base = {"effective_s": tm["effective_s"], "eval_s": tm["eval_s"], "save_s": tm["save_s"]} if tm else {}
    resumed_from = int(tm["global_step"]) if tm else 0
    if ck is None and (run_dir / "ledger.json").exists() and not train_done:
        print(f"[{tag}] earlier session left no usable checkpoint -> restarting this run from zero (the lost time is kept as 'wasted')")
    if ck is not None and not train_done:
        es = (read_json(ck / "trainer_state.json", {}) or {}).get("stateful_callbacks", {})
        print(f"[{tag}] RESUMING from {ck.name} (time stored in checkpoint: {tm['effective_s']:.0f}s); "
              f"early-stopping state in checkpoint: {es.get('EarlyStoppingCallback', 'not stored by this transformers version')}")

    args = build_training_args(spec, seed, run_dir, epochs, precision)
    meta = {"model": model_name, "variant": variant, "seed": seed, "gpu": GPU_NAME, "precision": precision}
    timer = RunTimer(run_dir, base=base, resumed_from_step=resumed_from, meta=meta, heartbeat_s=heartbeat_s)
    cbs = [EarlyStoppingCallback(early_stopping_patience=patience), timer]
    if crash_at_step:
        marker = run_dir / "crash_marker"
        class _CrashOnce(TrainerCallback):
            def on_step_end(self, args, state, control, **kw):
                if state.global_step >= crash_at_step and not marker.exists():
                    marker.write_text("crashed")
                    raise RuntimeError("SIMULATED CRASH (smoke test)")
                return control
        cbs.append(_CrashOnce())
    tparams = inspect.signature(Trainer.__init__).parameters
    tk = dict(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
              data_collator=DataCollatorWithPadding(tokenizer=tok), compute_metrics=compute_metrics_hf, callbacks=cbs)
    tk["processing_class" if "processing_class" in tparams else "tokenizer"] = tok
    trainer = TimedTrainer(**tk)
    trainer.timer = timer

    if train_done:
        st = _last_state(run_dir)
        best = st.get("best_model_checkpoint") if st else None
        print(f"[{tag}] training had already finished -> loading best checkpoint {best} for evaluation (no extra training)")
        trainer.model = AutoModelForSequenceClassification.from_pretrained(best).to(args.device)
        history = st.get("log_history", [])
        summary = read_json(run_dir / "ledger.json", {}).get("summary") or read_json(run_dir / "train_complete.json", {})
    else:
        try:
            trainer.train(resume_from_checkpoint=str(ck) if ck else None)
        except Exception:
            if not simulate_kill:
                timer.close("crashed")            # a real kill cannot do this; the heartbeat then holds the last known time
            raise
        history = trainer.state.log_history
        summary = read_json(run_dir / "ledger.json")["summary"]

    # ---- predictions (not part of the training time) ---------------------------------
    val_logits = _logits(trainer.predict(ds["validation"]))
    test_logits = _logits(trainer.predict(ds["test"]))
    val_y = [int(v) for v in ds["validation"]["label"]]
    val_pred = [int(v) for v in np.argmax(val_logits, axis=-1)]
    test_probs = softmax_rows(test_logits)
    test_pred = [int(np.argmax(p)) for p in test_probs]
    val_m, test_m = summarize(val_y, val_pred), summarize(TEST_Y, test_pred)
    collapsed = len(set(val_pred)) == 1                # model predicts one class for every validation answer
    write_test_probs(run_dir / "test_probs.csv", test_probs)
    atomic_write_json(run_dir / "history.json", history)

    evals = [h for h in history if "eval_loss" in h]
    best_ev = min(evals, key=lambda h: h["eval_loss"]) if evals else {}
    result = {
        "code_version": CODE_VERSION, "status": "done", "model": model_name, "variant": variant, "seed": seed,
        "checkpoint": spec["checkpoint"], "params_m": round(n_params / 1e6, 1), "precision": precision, "gpu": GPU_NAME,
        "dataset": DATASET_REPORT["path"], "epochs_run": len(evals), "best_epoch": best_ev.get("epoch"),
        "best_val_loss": best_ev.get("eval_loss"), "collapsed": collapsed,
        "timing": summary, "val": val_m, "test": test_m,
        "hf_train_runtime_last_session_s": next((h.get("train_runtime") for h in reversed(history) if "train_runtime" in h), None),
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
        "finished_utc": utc_now(),
    }

    # ---- weights: keep best model for the benchmark seed, drop everything else -----------
    if keep_weights:
        trainer.save_model(str(run_dir / "final_model"))
        tok.save_pretrained(str(run_dir / "final_model"))
        result["weights_mb"] = round(sum(f.stat().st_size for f in (run_dir / "final_model").glob("*")
                                         if f.suffix in (".safetensors", ".bin")) / 1e6, 1)
    atomic_write_json(run_dir / "result.json", result)         # completion marker, written BEFORE the cleanup below
    for c in list_checkpoints(run_dir):
        shutil.rmtree(c, ignore_errors=True)

    t = summary
    print(f"[{tag}] DONE  test acc {test_m['accuracy']:.2f}  BPD sens {test_m['bpd_sensitivity']:.2f}  | "
          f"train time effective {t['effective_s']:.0f}s, gross {t['gross_s']:.0f}s, wasted {t['wasted_s']:.0f}s, "
          f"sessions {t['n_sessions']}{'  (resumed)' if t['resumed'] else ''}  | epochs {len(evals)}"
          f"{'  | !!! COLLAPSED (single class)' if collapsed else ''}")
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 6. Smoke test: does the timer survive a crash?

Trains DistilBERT on a small subset three times: (A) is killed in the middle of epoch 2 *without any clean-up* (like Kaggle dying), (B) resumes and finishes, (C) is an uninterrupted reference run.
Expected: B resumes (2 sessions), `wasted > 0`, and B's **effective** time is close to C's. If this fails the notebook stops, so no hours are spent on wrong timings. Takes a few minutes; it is skipped once it has passed.

In [ ]:
SMOKE_FLAG = WORK_DIR / "smoke_passed.json"

def run_smoke_test():
    root = WORK_DIR / "smoke"
    shutil.rmtree(root, ignore_errors=True)
    sub = {"train": 800, "val": 120}
    steps_per_epoch = sub["train"] // (MODEL_SPECS["DistilBERT"]["bs"] * MODEL_SPECS["DistilBERT"]["accum"])
    crash_at = int(steps_per_epoch * 1.5)
    kw = dict(model_name="DistilBERT", seed=1, run_root=root, epochs=4, patience=99, subset=sub, heartbeat_s=1.0, keep_weights=False)

    print(">>> Smoke test A: first session, will be killed at step", crash_at)
    try:
        run_training(crash_at_step=crash_at, simulate_kill=True, **kw)
        raise AssertionError("the simulated crash did not happen")
    except RuntimeError as e:
        print("    crashed as planned:", e)
    print(">>> Smoke test B: resume")
    B = run_training(crash_at_step=crash_at, simulate_kill=True, **kw)
    print(">>> Smoke test C: uninterrupted reference")
    C = run_training(**{**kw, "seed": 2})

    tb, tc = B["timing"], C["timing"]
    checks = {
        "B was resumed in a 2nd session": tb["resumed"] and tb["n_sessions"] == 2,
        "B has wasted time (lost partial epoch)": tb["wasted_s"] > 0,
        "B effective < B gross": tb["effective_s"] < tb["gross_s"],
        "B effective close to uninterrupted C (+-40% or 8 s)": abs(tb["effective_s"] - tc["effective_s"]) <= max(0.4 * tc["effective_s"], 8.0),
        "B first session was marked interrupted": B["timing"]["session_status"][0] == "interrupted",
    }
    print("\n   effective  gross  wasted | B(resumed) %.1f %.1f %.1f | C(clean) %.1f %.1f %.1f" % (
        tb["effective_s"], tb["gross_s"], tb["wasted_s"], tc["effective_s"], tc["gross_s"], tc["wasted_s"]))
    for k, v in checks.items():
        print(("   PASS  " if v else "   FAIL  ") + k)
    if not all(checks.values()):
        raise AssertionError("Smoke test failed - do not start the long run. Check the ledger in " + str(root))
    atomic_write_json(SMOKE_FLAG, {"code_version": CODE_VERSION, "passed_utc": utc_now(), "checks": checks,
                                   "B": tb, "C": tc})
    shutil.rmtree(root, ignore_errors=True)
    print("\nSMOKE TEST PASSED")

_flag = read_json(SMOKE_FLAG, None)
if RUN_SMOKE_TEST and not (_flag and _flag.get("code_version") == CODE_VERSION):
    run_smoke_test()
elif RUN_SMOKE_TEST:
    print("Smoke test already passed earlier (", _flag["passed_utc"], ") - skipped.")

## 7. Main training: seeds x models

Order: seed 42 for all models first (so a complete result exists early), then seeds 43 and 44.
If DeBERTa-v3 collapses to a single class under fp16 (it did in earlier runs), it is retried **once in fp32**; both rows are kept and the fp32 run is clearly labelled `DeBERTa-fp32`.

In [ ]:
def status_table():
    rows = []
    for seed in SEEDS:
        for m in MODELS + ["DeBERTa-fp32"]:
            d = RUNS_DIR / f"seed{seed}" / m
            if not d.exists():
                continue
            r = read_json(d / "result.json", None)
            if r and r.get("code_version") == CODE_VERSION:
                rows.append((seed, m, "done", f"{r['timing']['effective_s']:.0f}s", f"{r['test']['accuracy']:.2f}"))
            else:
                led = read_json(d / "ledger.json", {})
                rows.append((seed, m, "partial (will resume)", f"{sum(s.get('active_s',0) for s in led.get('sessions',[])):.0f}s so far", "-"))
    return rows

def run_all():
    failed = []
    for seed in SEEDS:
        for m in MODELS:
            try:
                res = run_training(m, seed)
                if m == "DeBERTa" and res.get("collapsed") and PRECISION == "fp16":
                    print(f"[seed{seed}/DeBERTa] collapsed under fp16 -> one documented retry in fp32")
                    run_training(m, seed, variant="-fp32", precision="fp32")
            except Exception as e:
                print(f"\n!!! seed{seed}/{m} failed: {type(e).__name__}: {e}")
                traceback.print_exc()
                failed.append((seed, m, str(e)))
    print("\nRun status:")
    for r in status_table():
        print("  ", *r)
    if failed:
        print("\nFAILED runs (run this cell again: they resume from their last checkpoint):", failed)
    return failed

if RUN_MAIN_TRAINING:
    _failed = run_all()

## 8. Results: tables for the paper

In [ ]:
def load_results():
    out = []
    for p in sorted(RUNS_DIR.glob("seed*/*/result.json")):
        r = read_json(p, None)
        if r and r.get("code_version") == CODE_VERSION and r.get("status") == "done":
            probs = list(csv.DictReader(open(p.parent / "test_probs.csv")))
            score = [float(x["p_bpd"]) for x in probs]
            yb = [int(x["label"]) == 1 for x in probs]
            r["auroc_bpd"] = auroc(score, yb)
            r["spec_at_sens85"], _ = spec_at_sens(score, yb, 0.85)
            out.append(r)
    return out

def write_tables():
    R = load_results()
    if not R:
        print("no finished runs yet"); return
    # per run
    with open(WORK_DIR / "results_all_runs.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["model", "seed", "precision", "params_m", "epochs_run", "best_epoch", "collapsed",
                    "train_effective_s", "train_gross_s", "train_wasted_s", "eval_s", "save_s", "n_sessions", "resumed", "hf_last_session_runtime_s",
                    "test_acc", "test_macro_f1", "test_weighted_f1", "test_bpd_f1", "test_bpd_sens", "test_incon_recall",
                    "bin_spec", "bin_ppv", "bin_npv", "bin_acc", "auroc_bpd", "spec_at_sens85", "val_acc", "gpu", "weights_mb"])
        for r in R:
            t, m = r["timing"], r["test"]
            w.writerow([r["model"] + r["variant"], r["seed"], r["precision"], r["params_m"], r["epochs_run"], r["best_epoch"], r["collapsed"],
                        t["effective_s"], t["gross_s"], t["wasted_s"], t["eval_s"], t["save_s"], t["n_sessions"], t["resumed"], r["hf_train_runtime_last_session_s"],
                        round(m["accuracy"], 2), round(m["macro_f1"], 2), round(m["weighted_f1"], 2), round(m["bpd_f1"], 2), round(m["bpd_sensitivity"], 2),
                        round(m["inconclusive_recall"], 2), round(m["bin_specificity"], 2), round(m["bin_ppv"], 2), round(m["bin_npv"], 2),
                        round(m["bin_accuracy"], 2), round(r["auroc_bpd"], 4), round(r["spec_at_sens85"], 2), round(r["val"]["accuracy"], 2), r["gpu"], r.get("weights_mb")])
    # summary over seeds
    names = sorted({r["model"] + r["variant"] for r in R}, key=lambda n: (MODELS + ["DeBERTa-fp32"]).index(n) if n in MODELS + ["DeBERTa-fp32"] else 99)
    lines = []
    hdr = ["model", "seeds", "train time effective (s)", "gross (s)", "wasted (s)", "epochs", "accuracy", "macro F1", "BPD sens", "specificity", "AUROC(BPD)", "spec@sens85"]
    lines.append(hdr)
    for n in names:
        rs = [r for r in R if r["model"] + r["variant"] == n]
        f = lambda g, d=1: "%.*f ± %.*f" % ((d,) + pct_stats([g(r) for r in rs])[:1] + (d,) + pct_stats([g(r) for r in rs])[1:])
        lines.append([n, str(len(rs)), f(lambda r: r["timing"]["effective_s"], 0), f(lambda r: r["timing"]["gross_s"], 0), f(lambda r: r["timing"]["wasted_s"], 0),
                      f(lambda r: r["epochs_run"], 1), f(lambda r: r["test"]["accuracy"], 2), f(lambda r: r["test"]["macro_f1"], 2),
                      f(lambda r: r["test"]["bpd_sensitivity"], 2), f(lambda r: r["test"]["bin_specificity"], 2),
                      f(lambda r: r["auroc_bpd"] * 100, 1), f(lambda r: r["spec_at_sens85"], 1)])
    with open(WORK_DIR / "summary_mean_sd.csv", "w", newline="") as fh:
        csv.writer(fh).writerows(lines)
    widths = [max(len(str(l[i])) for l in lines) for i in range(len(hdr))]
    print("\nSUMMARY (mean ± sd over seeds; time in seconds; GPU:", GPU_NAME, ")\n")
    for l in lines:
        print("  ".join(str(c).ljust(widths[i]) for i, c in enumerate(l)))
    return R

RESULTS = write_tables()

## 9. Inference benchmark (same session, same GPU as training)

Per-response latency **including tokenisation**, measured with warm-up and `torch.cuda.synchronize()`. Models are loaded in fp32 (so weights size = parameters x 4 bytes).
* GPU, batch size 1 (one answer at a time, like a clinic tool)
* GPU, batch size 32 (time per answer)
* CPU, batch size 1 (the resource-constrained case), first 100 test answers

In [ ]:
def bench_model(model_name, run_dir, n_cpu=100):
    mdir = run_dir / "final_model"
    tok = AutoTokenizer.from_pretrained(str(mdir))
    model = AutoModelForSequenceClassification.from_pretrained(str(mdir)).eval()
    Q, A = TEST_Q_NORM, TEST_A_NORM
    def fwd(q, a, dev):
        enc = tok(q, a, truncation=True, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            return model(**enc).logits
    def stats(ms):
        ms = sorted(ms); return {"mean_ms": statistics.mean(ms), "median_ms": statistics.median(ms), "p95_ms": ms[int(0.95 * (len(ms) - 1))]}
    res = {"model": model_name, "weights_mb": sum(f.stat().st_size for f in mdir.glob("*") if f.suffix in (".safetensors", ".bin")) / 1e6}
    if torch.cuda.is_available():
        model.to("cuda")
        for i in range(20):
            fwd([Q[i]], [A[i]], "cuda")
        ms = []
        for q, a in zip(Q, A):
            torch.cuda.synchronize(); t = time.perf_counter()
            fwd([q], [a], "cuda")
            torch.cuda.synchronize(); ms.append((time.perf_counter() - t) * 1000)
        res.update({f"gpu_bs1_{k}": v for k, v in stats(ms).items()})
        fwd(Q[:32], A[:32], "cuda"); torch.cuda.synchronize()
        t = time.perf_counter()
        for i in range(0, len(Q), 32):
            fwd(Q[i:i + 32], A[i:i + 32], "cuda")
        torch.cuda.synchronize()
        res["gpu_bs32_ms_per_answer"] = (time.perf_counter() - t) * 1000 / len(Q)
    model.to("cpu")
    for i in range(5):
        fwd([Q[i]], [A[i]], "cpu")
    ms = []
    for q, a in zip(Q[:n_cpu], A[:n_cpu]):
        t = time.perf_counter(); fwd([q], [a], "cpu"); ms.append((time.perf_counter() - t) * 1000)
    res.update({f"cpu_bs1_{k}": v for k, v in stats(ms).items()})
    res["cpu_threads"] = torch.get_num_threads()
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

if RUN_INFERENCE_BENCH:
    bench_seed = KEEP_WEIGHTS_SEEDS[0]
    out_path = WORK_DIR / "inference_benchmark.csv"
    rows = []
    for m in MODELS:
        d = RUNS_DIR / f"seed{bench_seed}" / m
        if (d / "final_model").exists():
            print("benchmarking", m, "...")
            rows.append(bench_model(m, d))
    if rows:
        keys = sorted({k for r in rows for k in r}, key=lambda k: (k != "model", k))
        with open(out_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
            for r in rows:
                w.writerow({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()})
        print("\n%-12s %10s %14s %16s %14s" % ("model", "weights MB", "GPU bs1 (ms)", "GPU bs32 (ms/ans)", "CPU bs1 (ms)"))
        for r in rows:
            print("%-12s %10.1f %14.2f %16.2f %14.2f" % (r["model"], r["weights_mb"], r.get("gpu_bs1_median_ms", float("nan")),
                                                       r.get("gpu_bs32_ms_per_answer", float("nan")), r["cpu_bs1_median_ms"]))
    else:
        print("No saved weights found for seed", bench_seed)

## 10. VBRI benchmark (same session, same hardware)

Frozen `BAAI/bge-base-en-v1.5`, nine per-criterion databases built from the **same 4,050 training answers** the baselines were trained on, similarity-weighted k-NN vote (answers are used as raw text, as in the original VBRI experiments).
This part measures: index-build time, per-query latency (encode + search) on GPU and CPU, index size, and accuracy. **k is chosen on the validation set** (not on the test set); k = 5 is also reported for comparison.

In [ ]:
def vbri_benchmark():
    from sentence_transformers import SentenceTransformer
    ans = lambda ds: [str(a).strip() for a in ds["answer"]]
    tr_a, tr_q, tr_y = ans(RAW["train"]), [int(v) for v in RAW["train"]["question_id"]], [int(v) for v in RAW["train"]["label"]]
    va_a, va_q, va_y = ans(RAW["validation"]), [int(v) for v in RAW["validation"]["question_id"]], [int(v) for v in RAW["validation"]["label"]]
    te_a = ans(RAW["test"])
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    t = time.perf_counter(); bge = SentenceTransformer("BAAI/bge-base-en-v1.5", device=dev); load_s = time.perf_counter() - t

    def sync():
        if dev == "cuda": torch.cuda.synchronize()
    def encode(texts, bs=64):
        return bge.encode(texts, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    encode(tr_a[:64]); sync()                                  # warm-up
    builds = []
    for _ in range(3):
        sync(); t = time.perf_counter(); E = encode(tr_a); sync(); builds.append(time.perf_counter() - t)
    stores = {}
    for q in sorted(set(tr_q)):
        idx = [i for i, qq in enumerate(tr_q) if qq == q]
        stores[q] = (E[idx], np.array([tr_y[i] for i in idx]))
    index_mb = sum(s[0].nbytes + s[1].nbytes for s in stores.values()) / 1e6

    def vote(emb, q, k):
        M, L = stores[q]
        sims = M @ emb
        top = np.argpartition(-sims, k)[:k]
        sc = np.zeros(3)
        for j in top:
            sc[L[j]] += max(float(sims[j]), 0.0)
        return sc

    # --- k chosen on the VALIDATION set ---
    Ev = encode(va_a)
    val_f1 = {}
    for k in (1, 3, 5, 7, 9):
        pred = [int(np.argmax(vote(Ev[i], va_q[i], k))) for i in range(len(va_a))]
        val_f1[k] = summarize(va_y, pred)["macro_f1"]
    best_k = max(val_f1, key=lambda k: (val_f1[k], -k))
    print("validation macro-F1 by k:", {k: round(v, 2) for k, v in val_f1.items()}, "-> chosen k =", best_k)

    Et = encode(te_a)
    res = {"dataset": DATASET_REPORT["path"], "model_load_s": load_s, "index_build_s_runs": builds,
           "index_build_s_median": statistics.median(builds), "index_mb": index_mb, "n_vectors": int(E.shape[0]),
           "val_macro_f1_by_k": val_f1, "chosen_k": best_k}
    for k in sorted({best_k, 5}):
        sc = [vote(Et[i], TEST_QIDS[i], k) for i in range(len(te_a))]
        pred = [int(np.argmax(s)) for s in sc]
        res[f"test_k{k}"] = summarize(TEST_Y, pred)
        tot = [s.sum() or 1.0 for s in sc]
        pb = [float(s[1] / z) for s, z in zip(sc, tot)]
        yb = [y == 1 for y in TEST_Y]
        res[f"test_k{k}"]["auroc_bpd"] = auroc(pb, yb)
        res[f"test_k{k}"]["spec_at_sens85"] = spec_at_sens(pb, yb, 0.85)[0]
        with open(WORK_DIR / f"vbri_test_scores_k{k}.csv", "w", newline="") as f:
            w = csv.writer(f); w.writerow(["id", "question_id", "label", "p_normal", "p_bpd", "p_inconclusive"])
            for pid, q, y, s, z in zip(TEST_IDS, TEST_QIDS, TEST_Y, sc, tot):
                w.writerow([pid, q, y, *[f"{v / z:.6f}" for v in s]])

    # --- per-query latency: text -> prediction (encode + search), one answer at a time ---
    def per_query(n, device_label):
        enc_ms, srch_ms = [], []
        for i in range(n):
            sync(); t0 = time.perf_counter()
            e = bge.encode([te_a[i]], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
            sync(); t1 = time.perf_counter()
            vote(e, TEST_QIDS[i], best_k)
            t2 = time.perf_counter()
            enc_ms.append((t1 - t0) * 1000); srch_ms.append((t2 - t1) * 1000)
        tot = sorted(a + b for a, b in zip(enc_ms, srch_ms))
        return {f"{device_label}_encode_ms_median": statistics.median(enc_ms), f"{device_label}_search_ms_median": statistics.median(srch_ms),
                f"{device_label}_total_ms_median": statistics.median(tot), f"{device_label}_total_ms_mean": statistics.mean(tot),
                f"{device_label}_total_ms_p95": tot[int(0.95 * (len(tot) - 1))], f"{device_label}_n": n}
    for i in range(10):
        bge.encode([te_a[i]], show_progress_bar=False)
    if dev == "cuda":
        res.update(per_query(len(te_a), "gpu_bs1"))
    bge.to("cpu"); dev = "cpu"
    res.update(per_query(100, "cpu_bs1"))
    res["cpu_threads"] = torch.get_num_threads()
    atomic_write_json(WORK_DIR / "vbri_benchmark.json", res)

    k = best_k; m = res[f"test_k{k}"]
    print(f"\nVBRI (k={k}, chosen on validation)  accuracy {m['accuracy']:.2f}  macro-F1 {m['macro_f1']:.2f}  BPD sens {m['bpd_sensitivity']:.2f}  spec {m['bin_specificity']:.2f}  AUROC {m['auroc_bpd']:.3f}")
    m5 = res["test_k5"]; print(f"VBRI (k=5)                           accuracy {m5['accuracy']:.2f}  macro-F1 {m5['macro_f1']:.2f}  BPD sens {m5['bpd_sensitivity']:.2f}")
    print(f"index build {res['index_build_s_median']:.2f}s for {res['n_vectors']} answers (model load {load_s:.1f}s, not counted) | index {index_mb:.1f} MB")
    for lab in ("gpu_bs1", "cpu_bs1"):
        if f"{lab}_total_ms_median" in res:
            print(f"{lab}: encode {res[lab + '_encode_ms_median']:.2f} ms + search {res[lab + '_search_ms_median']:.3f} ms = {res[lab + '_total_ms_median']:.2f} ms per answer (median)")
    return res

if RUN_VBRI_BENCH:
    VBRI = vbri_benchmark()

## 11. Package the results (weights and checkpoints are left out)

In [ ]:
def package():
    zpath = WORK_DIR.parent / "final_baseline_run_results.zip"
    n = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in WORK_DIR.rglob("*"):
            if p.is_dir() or "final_model" in p.parts or any(part.startswith("checkpoint-") for part in p.parts) or "smoke" in p.parts:
                continue
            z.write(p, p.relative_to(WORK_DIR.parent)); n += 1
    print(f"Wrote {zpath} ({n} files, {zpath.stat().st_size/1e6:.1f} MB). Download it from the Kaggle output panel.")

package()